<a href="https://colab.research.google.com/github/sadineniManushree/flyrank--internship__ml/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sadineniManushree/flyrank--internship__ml/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [17]:
!git clone https://github.com/sadineniManushree/flyrank--internship__ml.git
%cd flyrank--internship__ml

Cloning into 'flyrank--internship__ml'...
remote: Enumerating objects: 150, done.
remote: Counting objects: 100% (150/150), done.
remote: Compressing objects: 100% (107/107), done.
remote: Total 150 (delta 58), reused 92 (delta 27), pack-reused 0 (from 0)
Receiving objects: 100% (150/150), 1.86 MiB | 4.92 MiB/s, done.
Resolving deltas: 100% (58/58), done.
/content/flyrank--internship__ml/flyrank--internship__ml/flyrank--internship__ml


## Decision Contract

- Goal: Identify pages with CTR opportunity that could benefit from
  snippet (title/meta description) updates.
- Success metric: CTR increase of at least X% after the fix is applied.
- Eligibility: Pages with at least [N] impressions in the last 90 days
  (enough data to trust the CTR number).
- Evaluation period: Recheck CTR [30/60] days after the snippet change,
  to allow enough time for the data to stabilize.

For every query, I figure out what CTR it should get based on its position (top spots normally get way more clicks). Then I compare that to what CTR it's actually getting. If a query has a lot of impressions (people are seeing it) but its real CTR is a lot lower than expected for its position, that's a missed opportunity — people are seeing it but not clicking, even though they should be. The bigger that gap, and the more impressions behind it, the higher the score — because fixing it (better title, meta description, etc.) could win a lot of extra clicks

In [18]:
import pandas as pd
import numpy as np

df = pd.read_csv('data/raw/content_refresh_anonymized.csv')
print(len(df), "rows")

30000 rows


In [19]:
import os

os.makedirs('work/outputs', exist_ok=True)

df['score'] = (df.groupby('position_tier')['ctr'].transform('mean') - df['ctr']) * df['impressions_90d']
df['reason_code'] = 'CTR_BELOW_EXPECTED'

threshold = df['score'].quantile(0.75)
df['action'] = df['score'].apply(lambda s: 'FIX_CTR' if s > threshold else 'MONITOR')

df_ranked = df.sort_values('score', ascending=False).reset_index(drop=True)
df_ranked.to_csv('work/outputs/baseline_action_score.csv', index=False)

print("Saved", len(df_ranked), "rows to work/outputs/baseline_action_score.csv")
print("Threshold used for FIX_CTR:", round(threshold, 2))
df_ranked.head(10)

Saved 30000 rows to work/outputs/baseline_action_score.csv
Threshold used for FIX_CTR: 568.22


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct,score,reason_code,action
0,content_8c19996aa890,client_4e07408562,70.0,0.01,LOW,0.00,keyword article,informational,2895.0,19343.0,...,11.73,16.03,0.00,excellent,top_3,down,-44.5,679143.820819,CTR_BELOW_EXPECTED,FIX_CTR
1,content_4c36c775b818,client_4e07408562,40.0,0.00,LOW,0.00,keyword article,informational,3097.0,20514.0,...,11.31,18.07,0.00,excellent,top_3,down,-33.2,497192.249268,CTR_BELOW_EXPECTED,FIX_CTR
2,content_8451fc6f034d,client_d029fa3a95,10.0,1.00,HIGH,14.11,keyword article,informational,3528.0,24856.0,...,2.02,2.26,2.02,excellent,top_3,up,73.1,395591.379371,CTR_BELOW_EXPECTED,FIX_CTR
3,content_5fe46e04994d,client_4e07408562,1900.0,0.00,LOW,0.00,keyword article,informational,NaN,NaN,...,4.23,26.90,0.38,excellent,page_1,down,-44.8,265311.627747,CTR_BELOW_EXPECTED,FIX_CTR
4,content_44e481c8f55b,client_19581e27de,20.0,0.26,LOW,0.43,keyword article,transactional,3040.0,21293.0,...,0.76,2.78,0.00,excellent,top_3,stable,-11.6,260665.005661,CTR_BELOW_EXPECTED,FIX_CTR
5,content_e12868d1f396,client_4e07408562,12100.0,0.01,LOW,1.36,keyword article,informational,2363.0,15540.0,...,5.94,11.97,0.00,excellent,top_3,stable,-14.8,211634.457079,CTR_BELOW_EXPECTED,FIX_CTR
6,content_aaef01a50def,client_19581e27de,4400.0,0.05,LOW,0.11,keyword article,informational,NaN,NaN,...,2.42,4.77,0.00,excellent,page_1,stable,-4.0,208119.083008,CTR_BELOW_EXPECTED,FIX_CTR
7,content_9532f197bbc8,client_4e07408562,10.0,0.00,LOW,0.00,keyword article,informational,NaN,NaN,...,8.01,28.75,0.00,excellent,top_3,down,-37.3,189723.461646,CTR_BELOW_EXPECTED,FIX_CTR
8,content_4a6607efcb46,client_6208ef0f77,0.0,0.00,LOW,0.00,keyword article,informational,4939.0,32266.0,...,2.30,6.11,2.30,excellent,top_3,up,5426.6,188722.351142,CTR_BELOW_EXPECTED,FIX_CTR
9,content_36ff89c8214e,client_19581e27de,0.0,0.00,LOW,0.00,keyword article,informational,NaN,NaN,...,1.68,4.48,0.00,excellent,page_1,stable,0.5,177786.075959,CTR_BELOW_EXPECTED,FIX_CTR


In [20]:
CTR_BELOW_EXPECTED = 'CTR_BELOW_EXPECTED'

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [21]:
import os

os.makedirs('work/outputs', exist_ok=True)

# 1. Score = (expected CTR for that position tier - actual CTR) x impressions
df['score'] = (df.groupby('position_tier')['ctr'].transform('mean') - df['ctr']) * df['impressions_90d']

# 2. Reason code - same label for every flagged row
df['reason_code'] = 'CTR_BELOW_EXPECTED'

# 3. Action label - top 25% of scores get FIX_CTR, rest get MONITOR
threshold = df['score'].quantile(0.75)
df['action'] = df['score'].apply(lambda s: 'FIX_CTR' if s > threshold else 'MONITOR')

# 4. Rank everything - highest score (biggest opportunity) first
df_ranked = df.sort_values('score', ascending=False).reset_index(drop=True)

# 5. Write the CSV
df_ranked.to_csv('work/outputs/baseline_action_score.csv', index=False)

print("Saved", len(df_ranked), "rows to work/outputs/baseline_action_score.csv")
print("Threshold used for FIX_CTR:", round(threshold, 2))
df_ranked.head(10)

Saved 30000 rows to work/outputs/baseline_action_score.csv
Threshold used for FIX_CTR: 568.22


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct,score,reason_code,action
0,content_8c19996aa890,client_4e07408562,70.0,0.01,LOW,0.00,keyword article,informational,2895.0,19343.0,...,11.73,16.03,0.00,excellent,top_3,down,-44.5,679143.820819,CTR_BELOW_EXPECTED,FIX_CTR
1,content_4c36c775b818,client_4e07408562,40.0,0.00,LOW,0.00,keyword article,informational,3097.0,20514.0,...,11.31,18.07,0.00,excellent,top_3,down,-33.2,497192.249268,CTR_BELOW_EXPECTED,FIX_CTR
2,content_8451fc6f034d,client_d029fa3a95,10.0,1.00,HIGH,14.11,keyword article,informational,3528.0,24856.0,...,2.02,2.26,2.02,excellent,top_3,up,73.1,395591.379371,CTR_BELOW_EXPECTED,FIX_CTR
3,content_5fe46e04994d,client_4e07408562,1900.0,0.00,LOW,0.00,keyword article,informational,NaN,NaN,...,4.23,26.90,0.38,excellent,page_1,down,-44.8,265311.627747,CTR_BELOW_EXPECTED,FIX_CTR
4,content_44e481c8f55b,client_19581e27de,20.0,0.26,LOW,0.43,keyword article,transactional,3040.0,21293.0,...,0.76,2.78,0.00,excellent,top_3,stable,-11.6,260665.005661,CTR_BELOW_EXPECTED,FIX_CTR
5,content_e12868d1f396,client_4e07408562,12100.0,0.01,LOW,1.36,keyword article,informational,2363.0,15540.0,...,5.94,11.97,0.00,excellent,top_3,stable,-14.8,211634.457079,CTR_BELOW_EXPECTED,FIX_CTR
6,content_aaef01a50def,client_19581e27de,4400.0,0.05,LOW,0.11,keyword article,informational,NaN,NaN,...,2.42,4.77,0.00,excellent,page_1,stable,-4.0,208119.083008,CTR_BELOW_EXPECTED,FIX_CTR
7,content_9532f197bbc8,client_4e07408562,10.0,0.00,LOW,0.00,keyword article,informational,NaN,NaN,...,8.01,28.75,0.00,excellent,top_3,down,-37.3,189723.461646,CTR_BELOW_EXPECTED,FIX_CTR
8,content_4a6607efcb46,client_6208ef0f77,0.0,0.00,LOW,0.00,keyword article,informational,4939.0,32266.0,...,2.30,6.11,2.30,excellent,top_3,up,5426.6,188722.351142,CTR_BELOW_EXPECTED,FIX_CTR
9,content_36ff89c8214e,client_19581e27de,0.0,0.00,LOW,0.00,keyword article,informational,NaN,NaN,...,1.68,4.48,0.00,excellent,page_1,stable,0.5,177786.075959,CTR_BELOW_EXPECTED,FIX_CTR


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [22]:
pos_table = df.groupby('position_tier')['ctr'].agg(avg_ctr='mean', n='count')
print(pos_table)

                avg_ctr      n
position_tier                 
deep           0.150212   1319
page_1         0.652467  11814
page_3_5       0.222484   7242
striking       0.323239   7304
top_3          1.483611   2321


In [23]:
top20 = df_ranked.head(20)[['content_id', 'position_tier', 'ctr', 'impressions_90d', 'score', 'trend_direction', 'trend_pct', 'action']]
print(top20.to_string(index=False))

          content_id position_tier  ctr  impressions_90d         score trend_direction  trend_pct  action
content_8c19996aa890         top_3 0.15           509252 679143.820819            down      -44.5 FIX_CTR
content_4c36c775b818         top_3 0.41           463103 497192.249268            down      -33.2 FIX_CTR
content_8451fc6f034d         top_3 0.03           272144 395591.379371              up       73.1 FIX_CTR
content_5fe46e04994d        page_1 0.14           517715 265311.627747            down      -44.8 FIX_CTR
content_44e481c8f55b         top_3 0.65           312694 260665.005661          stable      -11.6 FIX_CTR
content_e12868d1f396         top_3 0.07           149712 211634.457079          stable      -14.8 FIX_CTR
content_aaef01a50def        page_1 0.25           517109 208119.083008          stable       -4.0 FIX_CTR
content_9532f197bbc8         top_3 0.87           309192 189723.461646            down      -37.3 FIX_CTR
content_4a6607efcb46         top_3 0.01       

## Top-20 Review

1. FIX_CTR | 509,252 impressions, ctr 0.15, trend DOWN -44.5% | HIGH confidence —
   huge impressions and CTR already trending down, strong agreement | Wrong if:
   the decline is industry-wide (seasonal dip), not specific to this page.

2. FIX_CTR | 463,103 impressions, ctr 0.41, trend DOWN -33.2% | HIGH confidence —
   large volume, real decline already visible | Wrong if: a recent algorithm
   update affected this whole content type, not just this page.

3. FIX_CTR | 272,144 impressions, ctr 0.03, trend UP +73.1% | MEDIUM confidence —
   very low CTR is a real opportunity, but trend is UP, contradicting the "decline"
   story | Wrong if: this page is actually improving on its own and doesn't need
   intervention — the flag may be premature.

4. FIX_CTR | 517,715 impressions, ctr 0.14, trend DOWN -44.8% | HIGH confidence —
   top volume in this batch, matches expected pattern well.

5. FIX_CTR | 312,694 impressions, ctr 0.65, trend STABLE -11.6% | MEDIUM
   confidence — CTR (0.65) isn't actually that low for top_3 | Wrong if: 0.65
   is within normal range for this page's specific query intent.

6. FIX_CTR | 149,712 impressions, ctr 0.07, trend STABLE -14.8% | HIGH
   confidence — very low CTR for a top_3 page, real gap.

7. FIX_CTR | 517,109 impressions, ctr 0.25, trend STABLE -4.0% | MEDIUM
   confidence — decent volume but trend is nearly flat, not clearly worsening |
   Wrong if: this page's CTR is already at a stable equilibrium, not a solvable gap.

8. FIX_CTR | 309,192 impressions, ctr 0.87, trend DOWN -37.3% | LOW confidence —
   CTR (0.87) is very high already; flagging this as a "CTR problem" looks off |
   Wrong if: the score is being driven by expected-CTR miscalibration for top_3,
   not a real gap — worth double-checking the math on this one.

9. FIX_CTR | 128,068 impressions, ctr 0.01, trend UP +5426.6% | LOW confidence —
   that trend_pct is almost certainly a data artifact (huge % jump usually means
   dividing by a near-zero starting value) | Wrong if: this is just noise from a
   tiny baseline traffic number — don't trust the trend figure here at all.

10. FIX_CTR | 295,097 impressions, ctr 0.05, trend STABLE +0.5% | HIGH
    confidence — very low CTR, high volume, real opportunity.

11. FIX_CTR | 416,180 impressions, ctr 0.23, trend DOWN -27.0% | HIGH
    confidence — solid volume and consistent decline.

12. FIX_CTR | 345,111 impressions, ctr 0.21, trend UP +556.2% | LOW confidence —
    trend is sharply UP, contradicts the "opportunity" framing | Wrong if: this
    page is already recovering and doesn't need a fix.

13. FIX_CTR | 309,910 impressions, ctr 0.16, trend DOWN -41.8% | HIGH
    confidence — good agreement between low CTR and real decline.

14. FIX_CTR | 223,271 impressions, ctr 0.03, trend STABLE +17.2% | MEDIUM
    confidence — CTR is very low (good signal) but trend is flat/slightly up |
    Wrong if: this page never had strong CTR to begin with (query mismatch),
    not something a snippet fix will solve.

15. FIX_CTR | 208,678 impressions, ctr 0.00, trend DOWN -43.4% | HIGH
    confidence — CTR essentially zero, strong decline, clear case.

16. FIX_CTR | 123,561 impressions, ctr 0.41, trend STABLE +19.9% | LOW
    confidence — CTR isn't dramatically low for top_3, trend slightly improving |
    Wrong if: this page doesn't actually need attention right now.

17. FIX_CTR | 160,959 impressions, ctr 0.69, trend STABLE -5.0% | LOW
    confidence — CTR (0.69) looks healthy; questionable flag | Wrong if: the
    expected-CTR baseline for top_3 is set too high, over-flagging fine pages.

18. FIX_CTR | 213,963 impressions, ctr 0.10, trend STABLE -11.6% | MEDIUM
    confidence — low CTR, moderate volume, plausible opportunity.

19. FIX_CTR | 201,111 impressions, ctr 0.11, trend STABLE -19.6% | MEDIUM
    confidence — similar profile to row 18, reasonable flag.

20. FIX_CTR | 131,328 impressions, ctr 0.70, trend STABLE -12.5% | LOW
    confidence — CTR (0.70) is healthy for top_3; likely a weak flag | Wrong if:
    the position-tier average is being skewed by outliers, making 0.70 look
    artificially low relative to its group.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

## Weak Picks

Several picks in the top 20 look questionable because their actual CTR is
already healthy, even though the score flagged them:

- **Row 8** (ctr 0.87, top_3, score 189,723) — a CTR of 0.87 is very strong.
  Flagging this as a "CTR problem" doesn't make sense on its face. This
  suggests the expected-CTR average for `top_3` may be getting pulled up by
  a small number of extremely high performers, making otherwise-good pages
  look like gaps by comparison.
- **Row 17** (ctr 0.69, top_3, score 127,738) and **Row 20** (ctr 0.70,
  top_3, score 102,910) — same pattern: healthy CTR, still flagged, likely
  for the same reason as row 8.
- **Row 9** (trend_pct +5426.6%) — this number is almost certainly a data
  artifact, not a real signal. A percentage change that large usually comes
  from dividing by a starting value close to zero. I would not trust this
  row's trend figure at all, and would exclude extreme trend_pct outliers
  like this from any confidence judgment.

**Root cause worth investigating:** the `position_tier` groups (e.g.
`top_3`) may be too broad, meaning the "expected CTR" average within a tier
is skewed by a few standout pages, which then makes moderately strong pages
look artificially like opportunities. A more precise position bucket (or
using `avg_position` directly instead of the tier) might fix this in a
future version of the rule.

## Leakage Check

I reviewed the code that builds `score`, `reason_code`, and `action`. It
uses only observable, pre-decision columns: `position_tier`, `ctr`, and
`impressions_90d`. It does not reference any FlyRank product-decision
columns (there is no `health_score`, `needs_ctr_fix`, `is_quick_win`,
`priority_score`, or `action_type` column in this dataset at all, so none
could have leaked in even by accident).

The `trend_direction` and `trend_pct` columns were used only in the
top-20 *review* (to sanity-check whether flagged pages are actually
declining), never as an input to the score itself — so there's no
circularity between the rule and the outcome I used to judge it. The score
is calculated purely from current-state signals available at decision
time.

## Honest Measurement

This baseline only flags opportunities — it does not yet measure whether
fixes actually worked. To do that honestly, before making any change to a
flagged page, I would record its current CTR as the "before" value. After
applying a fix (e.g., updating the title or meta description), I would wait
a defined evaluation period (e.g., 30 days) before checking CTR again, to
let the data stabilize and avoid judging results too early. Only after that
window would I compare the "after" CTR to the "before" CTR to confirm
whether the fix actually improved performance.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.